In [1]:
# Install PySpark (Colab does not have it pre-installed)
!pip install pyspark -q


In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("CSE488-Lab04-FrequentPatternMining") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
spark


---
## 5. Dataset for Independent Project Work


**Groceries dataset (Kaggle, no competition login needed):** https://www.kaggle.com/datasets/irfanasrullah/groceries — ~9,835 transactions, ~170 items, small enough to run in seconds; good for a quicker in-class exercise rather than a full project.


In [3]:
import pandas as pd

raw = pd.read_csv('groceries-groceries.csv')
print(raw.shape)
raw.head()

(9835, 33)


,Item(s),Item 1,Item 2,Item 3,Item 4,Item 5,Item 6,Item 7,Item 8,Item 9,...,Item 23,Item 24,Item 25,Item 26,Item 27,Item 28,Item 29,Item 30,Item 31,Item 32
0,4,citrus fruit,semi-finished bread,margarine,ready soups,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,3,tropical fruit,yogurt,coffee,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1,whole milk,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,pip fruit,yogurt,cream cheese,meat spreads,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4,other vegetables,whole milk,condensed milk,long life bakery product,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
data = raw.drop(columns=['Item(s)']).stack().reset_index(level=1, drop=True).to_frame('item')
data.index.name = 'transaction_id'

df_groceries = spark.createDataFrame(data.reset_index())

df_groceries_baskets = df_groceries.groupBy('transaction_id').agg(F.collect_list('item').alias('items'))

df_groceries_baskets_cleaned = df_groceries_baskets.dropna()
df_groceries_baskets_final = df_groceries_baskets_cleaned.filter(F.size('items') > 1)

df_groceries_baskets_final.show(5, truncate=False)
print("Number of grocery baskets:", df_groceries_baskets_final.count())

+--------------+------------------------------------------------------------------------+
|transaction_id|items                                                                   |
+--------------+------------------------------------------------------------------------+
|0             |[citrus fruit, semi-finished bread, margarine, ready soups]             |
|1             |[tropical fruit, yogurt, coffee]                                        |
|3             |[pip fruit, yogurt, cream cheese, meat spreads]                         |
|4             |[other vegetables, whole milk, condensed milk, long life bakery product]|
|5             |[whole milk, butter, yogurt, rice, abrasive cleaner]                    |
+--------------+------------------------------------------------------------------------+
only showing top 5 rows
Number of grocery baskets: 7676


In [6]:
from pyspark.ml.fpm import FPGrowth
fpGrowth_retail = FPGrowth(itemsCol="items", minSupport=0.02, minConfidence=0.3)
model_retail = fpGrowth_retail.fit(df_groceries_baskets_final)


In [ ]:
print("Number of frequent itemsets:", model_retail.freqItemsets.count())
model_retail.freqItemsets.orderBy(F.desc("freq")).show(15, truncate=60)


Number of frequent itemsets: 172
+------------------------------+----+
|                         items|freq|
+------------------------------+----+
|                  [whole milk]|2392|
|            [other vegetables]|1841|
|                  [rolls/buns]|1700|
|                        [soda]|1559|
|                      [yogurt]|1332|
|             [root vegetables]|1047|
|               [bottled water]|1020|
|              [tropical fruit]|1009|
|               [shopping bags]| 921|
|                     [sausage]| 900|
|                      [pastry]| 838|
|                [citrus fruit]| 798|
|[other vegetables, whole milk]| 736|
|                  [newspapers]| 731|
|                   [pip fruit]| 716|
+------------------------------+----+
only showing top 15 rows


In [7]:
print("Number of association rules:", model_retail.associationRules.count())
model_retail.associationRules.orderBy(F.desc("lift")).show(15, truncate=60)

Number of association rules: 58
+------------------------------+------------------+-------------------+------------------+--------------------+
|                    antecedent|        consequent|         confidence|              lift|             support|
+------------------------------+------------------+-------------------+------------------+--------------------+
|                        [beef]| [root vegetables]| 0.3489795918367347|2.5585170457867963|0.022277227722772276|
|[other vegetables, whole milk]| [root vegetables]|0.30978260869565216| 2.271147377600598|  0.0297029702970297|
| [root vegetables, whole milk]|[other vegetables]|0.47401247401247404|1.9763822653556495|  0.0297029702970297|
|                        [curd]|          [yogurt]| 0.3300970873786408| 1.902271203242077| 0.02214695153725899|
|             [root vegetables]|[other vegetables]|0.44508118433619864|1.8557540309422385| 0.06070870244919229|
|                     [chicken]|[other vegetables]|0.43243243243243246| 